In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = "../data/Processed/furniture_returns_featured.xlsx"

df = pd.read_excel(file_path)

print("Feature engineered dataset loaded successfully!")
print("Dataset shape:", df.shape)

Feature engineered dataset loaded successfully!
Dataset shape: (5058, 23)


In [3]:
df.head()

,Return ID,Product,Value ₹,Transit Days,Return Reason,Condition Hint,Severity,Safety,SLA hrs,Inspection Outcome,...,Value Loss %,Severity Score,Safety Score,Condition Risk Score,Transit Risk Score,SLA Urgency Score,Value Loss Risk Score,Priority Score,Priority,Priority Evidence
0,R-3047,bed,46133,2,size issue,no visible damage,Unknown,low,24,no defect,...,2.85,1,1,1,1,4,1,1.3,Low,Urgent SLA
1,R2886,bed,40964,2,damaged,torn fabric,moderate,high,48,repairable,...,29.93,1,1,1,1,3,3,1.8,Low,High resale value loss; Urgent SLA
2,R2747,Unknown,24559,1,damaged,chipped surface,moderate,low,48,minor damage,...,12.39,1,1,1,1,3,2,1.5,Low,Urgent SLA
3,R4670,dining table,26326,1,defective,loose joints,minor,low,48,minor damage,...,18.35,1,1,1,1,3,2,1.5,Low,Urgent SLA
4,R4898,dining table,26326,0,not as expected,color mismatch,minor,low,48,minor damage,...,38.85,1,1,1,1,3,3,1.8,Low,High resale value loss; Urgent SLA


In [4]:
required_columns = [
    "Return ID",
    "Value Loss Risk Score",
    "Severity Score",
    "Safety Score",
    "Transit Risk Score",
    "SLA Urgency Score",
    "Condition Risk Score"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are available!")

All required columns are available!


In [5]:
df["Baseline Score"] = (
    df["Value Loss Risk Score"] * 0.30 +
    df["Severity Score"] * 0.20 +
    df["Safety Score"] * 0.20 +
    df["Transit Risk Score"] * 0.15 +
    df["SLA Urgency Score"] * 0.10 +
    df["Condition Risk Score"] * 0.05
)

df["Baseline Score"] = df["Baseline Score"].round(2)

print("Baseline scores calculated successfully!")
df[["Return ID", "Baseline Score"]].head(10)

Baseline scores calculated successfully!


,Return ID,Baseline Score
0,R-3047,1.3
1,R2886,1.8
2,R2747,1.5
3,R4670,1.5
4,R4898,1.8
5,R2596,2.0
6,R3729,1.8
7,R3755,1.4
8,R3354,2.0
9,r79,1.1


In [6]:
def assign_baseline_priority(score):
    
    if score >= 2.6:
        return "High"
    
    elif score >= 1.8:
        return "Medium"
    
    else:
        return "Low"

In [7]:
df["Baseline Priority"] = (
    df["Baseline Score"]
    .apply(assign_baseline_priority)
)

print(df["Baseline Priority"].value_counts())

Baseline Priority
Low       3933
Medium    1115
High        10
Name: count, dtype: int64


In [8]:
def create_baseline_evidence(row):
    
    reasons = []
    
    if row["Value Loss Risk Score"] >= 3:
        reasons.append("High value loss")
    
    if row["Severity Score"] >= 3:
        reasons.append("High damage severity")
    
    if row["Safety Score"] >= 3:
        reasons.append("High safety risk")
    
    if row["Transit Risk Score"] >= 3:
        reasons.append("Long transit duration")
    
    if row["SLA Urgency Score"] >= 3:
        reasons.append("Urgent SLA")
    
    if row["Condition Risk Score"] >= 3:
        reasons.append("Serious condition issue")
    
    if not reasons:
        reasons.append("Standard inspection requirement")
    
    return "; ".join(reasons)


df["Baseline Evidence"] = df.apply(
    create_baseline_evidence,
    axis=1
)

In [9]:
result_columns = [
    "Return ID",
    "Product",
    "Value Loss %",
    "Severity",
    "Safety",
    "Transit Days",
    "SLA hrs",
    "Baseline Score",
    "Baseline Priority",
    "Baseline Evidence"
]

df[result_columns].head(20)

,Return ID,Product,Value Loss %,Severity,Safety,Transit Days,SLA hrs,Baseline Score,Baseline Priority,Baseline Evidence
0,R-3047,bed,2.85,Unknown,low,2,24,1.30,Low,Urgent SLA
1,R2886,bed,29.93,moderate,high,2,48,1.80,Medium,High value loss; Urgent SLA
2,R2747,Unknown,12.39,moderate,low,1,48,1.50,Low,Urgent SLA
3,R4670,dining table,18.35,minor,low,1,48,1.50,Low,Urgent SLA
4,R4898,dining table,38.85,minor,low,0,48,1.80,Medium,High value loss; Urgent SLA
5,R2596,bed,51.89,severe,high,0,72,2.00,Medium,High value loss
6,R3729,sofa,42.80,severe,medium,0,48,1.80,Medium,High value loss; Urgent SLA
7,R3755,wardrobe,5.48,minor,medium,6,72,1.40,Low,Long transit duration
8,R3354,coffee table,29.25,moderate,Unknown,7,72,2.00,Medium,High value loss; Long transit duration
9,r79,chair,0.35,none,low,2,72,1.10,Low,Standard inspection requirement


In [10]:
priority_distribution = pd.DataFrame({
    "Count": df["Baseline Priority"].value_counts(),
    "Percentage": (
        df["Baseline Priority"]
        .value_counts(normalize=True) * 100
    ).round(2)
})

priority_distribution

,Count,Percentage
Baseline Priority,,
Low,3933,77.76
Medium,1115,22.04
High,10,0.20


In [11]:
high_priority_returns = df[
    df["Baseline Priority"] == "High"
]

high_priority_returns[
    [
        "Return ID",
        "Product",
        "Value ₹",
        "Value Loss %",
        "Severity",
        "Safety",
        "Transit Days",
        "SLA hrs",
        "Baseline Score",
        "Baseline Evidence"
    ]
].head(20)

,Return ID,Product,Value ₹,Value Loss %,Severity,Safety,Transit Days,SLA hrs,Baseline Score,Baseline Evidence
194,R1758,sofa,49846,57.35,moderate,high,16,24,2.65,High value loss; Long transit duration; Urgent...
602,R-1311,chair,26326,73.91,Unknown,low,11,24,2.65,High value loss; Long transit duration; Urgent...
940,R1732,bookshelf,10108,78.19,severe,low,16,24,2.65,High value loss; Long transit duration; Urgent...
1104,R4988,bed,47256,51.23,none,Unknown,18,24,2.65,High value loss; Long transit duration; Urgent...
1266,R3277,chair,7496,83.02,severe,medium,15,24,2.65,High value loss; Long transit duration; Urgent...
1450,r3577,bed,44660,50.86,minor,low,11,24,2.65,High value loss; Long transit duration; Urgent...
1684,R4548,bookshelf,10869,54.84,Unknown,low,11,24,2.65,High value loss; Long transit duration; Urgent...
2876,R3290,chair,8057,73.01,Unknown,Unknown,18,24,2.65,High value loss; Long transit duration; Urgent...
3603,R1247,coffee table,10765,67.67,minor,low,12,24,2.65,High value loss; Long transit duration; Urgent...
4435,R3194,bookshelf,9158,68.44,moderate,high,28,24,2.65,High value loss; Long transit duration; Urgent...


In [12]:
medium_priority_returns = df[
    df["Baseline Priority"] == "Medium"
]

medium_priority_returns[
    [
        "Return ID",
        "Product",
        "Value Loss %",
        "Baseline Score",
        "Baseline Evidence"
    ]
].head(20)

,Return ID,Product,Value Loss %,Baseline Score,Baseline Evidence
1,R2886,bed,29.93,1.80,High value loss; Urgent SLA
4,R4898,dining table,38.85,1.80,High value loss; Urgent SLA
5,R2596,bed,51.89,2.00,High value loss
6,R3729,sofa,42.80,1.80,High value loss; Urgent SLA
8,R3354,coffee table,29.25,2.00,High value loss; Long transit duration
12,R1245,wardrobe,35.93,2.00,High value loss; Long transit duration
15,R1382,coffee table,12.76,1.80,Long transit duration; Urgent SLA
18,R1261,bed,42.10,2.05,High value loss; Urgent SLA
21,R-3402,chair,17.42,1.85,Long transit duration
23,R2380,bed,59.91,2.35,High value loss; Urgent SLA


In [13]:
low_priority_returns = df[
    df["Baseline Priority"] == "Low"
]

low_priority_returns[
    [
        "Return ID",
        "Product",
        "Value Loss %",
        "Baseline Score",
        "Baseline Evidence"
    ]
].head(20)

,Return ID,Product,Value Loss %,Baseline Score,Baseline Evidence
0,R-3047,bed,2.85,1.30,Urgent SLA
2,R2747,Unknown,12.39,1.50,Urgent SLA
3,R4670,dining table,18.35,1.50,Urgent SLA
7,R3755,wardrobe,5.48,1.40,Long transit duration
9,r79,chair,0.35,1.10,Standard inspection requirement
10,R4559,bed,1.78,1.30,Urgent SLA
11,R2281,wardrobe,10.09,1.40,Standard inspection requirement
13,R4778,dining table,0.86,1.40,Long transit duration
14,R3301,dining table,13.57,1.65,Urgent SLA
16,R1821,wardrobe,9.17,1.50,Long transit duration; Urgent SLA


In [14]:
df.groupby("Baseline Priority")[
    "Baseline Score"
].agg(
    ["count", "mean", "min", "max"]
).round(2)

,count,mean,min,max
Baseline Priority,,,,
High,10,2.65,2.65,2.65
Low,3933,1.38,1.10,1.75
Medium,1115,2.02,1.80,2.55


In [15]:
prioritized_returns = df.sort_values(
    by="Baseline Score",
    ascending=False
)

prioritized_returns[
    [
        "Return ID",
        "Product",
        "Baseline Score",
        "Baseline Priority",
        "Baseline Evidence"
    ]
].head(20)

,Return ID,Product,Baseline Score,Baseline Priority,Baseline Evidence
1450,r3577,bed,2.65,High,High value loss; Long transit duration; Urgent...
2876,R3290,chair,2.65,High,High value loss; Long transit duration; Urgent...
194,R1758,sofa,2.65,High,High value loss; Long transit duration; Urgent...
3603,R1247,coffee table,2.65,High,High value loss; Long transit duration; Urgent...
1266,R3277,chair,2.65,High,High value loss; Long transit duration; Urgent...
4435,R3194,bookshelf,2.65,High,High value loss; Long transit duration; Urgent...
1104,R4988,bed,2.65,High,High value loss; Long transit duration; Urgent...
940,R1732,bookshelf,2.65,High,High value loss; Long transit duration; Urgent...
1684,R4548,bookshelf,2.65,High,High value loss; Long transit duration; Urgent...
602,R-1311,chair,2.65,High,High value loss; Long transit duration; Urgent...


In [16]:
df["Inspection Rank"] = (
    df["Baseline Score"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)

df[
    [
        "Return ID",
        "Baseline Score",
        "Baseline Priority",
        "Inspection Rank"
    ]
].sort_values("Inspection Rank").head(20)

,Return ID,Baseline Score,Baseline Priority,Inspection Rank
2876,R3290,2.65,High,1
4435,R3194,2.65,High,1
1266,R3277,2.65,High,1
3603,R1247,2.65,High,1
1684,R4548,2.65,High,1
1450,r3577,2.65,High,1
940,R1732,2.65,High,1
602,R-1311,2.65,High,1
1104,R4988,2.65,High,1
194,R1758,2.65,High,1


In [17]:
top_10 = df.sort_values(
    "Inspection Rank"
).head(10)

top_10[
    [
        "Inspection Rank",
        "Return ID",
        "Product",
        "Baseline Score",
        "Baseline Priority",
        "Baseline Evidence"
    ]
]

,Inspection Rank,Return ID,Product,Baseline Score,Baseline Priority,Baseline Evidence
2876,1,R3290,chair,2.65,High,High value loss; Long transit duration; Urgent...
4435,1,R3194,bookshelf,2.65,High,High value loss; Long transit duration; Urgent...
1266,1,R3277,chair,2.65,High,High value loss; Long transit duration; Urgent...
3603,1,R1247,coffee table,2.65,High,High value loss; Long transit duration; Urgent...
1684,1,R4548,bookshelf,2.65,High,High value loss; Long transit duration; Urgent...
1450,1,r3577,bed,2.65,High,High value loss; Long transit duration; Urgent...
940,1,R1732,bookshelf,2.65,High,High value loss; Long transit duration; Urgent...
602,1,R-1311,chair,2.65,High,High value loss; Long transit duration; Urgent...
1104,1,R4988,bed,2.65,High,High value loss; Long transit duration; Urgent...
194,1,R1758,sofa,2.65,High,High value loss; Long transit duration; Urgent...


In [18]:
output_path = "../data/Processed/furniture_returns_baseline.xlsx"

df.to_excel(
    output_path,
    index=False
)

print("Baseline prioritization results saved successfully!")
print(output_path)

Baseline prioritization results saved successfully!
../data/Processed/furniture_returns_baseline.xlsx


In [19]:
print("=" * 60)
print("BASELINE PRIORITIZER SUMMARY")
print("=" * 60)

print("\nTotal Returns:", len(df))

print("\nPriority Distribution:")
print(df["Baseline Priority"].value_counts())

print("\nTop 5 Returns for Inspection:")

print(
    df.sort_values("Inspection Rank")
    [
        [
            "Inspection Rank",
            "Return ID",
            "Product",
            "Baseline Priority",
            "Baseline Score"
        ]
    ]
    .head(5)
)

BASELINE PRIORITIZER SUMMARY

Total Returns: 5058

Priority Distribution:
Baseline Priority
Low       3933
Medium    1115
High        10
Name: count, dtype: int64

Top 5 Returns for Inspection:
      Inspection Rank Return ID       Product Baseline Priority  \
2876                1     R3290         chair              High   
4435                1     R3194     bookshelf              High   
1266                1     R3277         chair              High   
3603                1    R1247   coffee table              High   
1684                1     R4548     bookshelf              High   

      Baseline Score  
2876            2.65  
4435            2.65  
1266            2.65  
3603            2.65  
1684            2.65  
